In [ ]:
import os
import json
import shutil
import random

import numpy as np
import torch
import optuna
from ultralytics import YOLO

# ============================================================================
# IMPORT CENTRALIZED CONFIG - All paths defined in config.py
# ============================================================================
from config import (
    DATA_YAML, YOLOV8S_WEIGHTS, YOLO_OPTUNA_DIR,
    IMGSZ, BATCH, DEVICE,
    OPTUNA_N_TRIALS, OPTUNA_EPOCHS_PER_TRIAL, OPTUNA_FINAL_EPOCHS
)

# Convert Path objects to strings for compatibility
DATA_YAML = str(DATA_YAML)
SAVE_DIR = str(YOLO_OPTUNA_DIR)
WEIGHTS = str(YOLOV8S_WEIGHTS)

os.makedirs(SAVE_DIR, exist_ok=True)

# Optuna search settings (from config)
N_TRIALS = OPTUNA_N_TRIALS
EPOCHS_PER_TRIAL = OPTUNA_EPOCHS_PER_TRIAL
FINAL_EPOCHS = OPTUNA_FINAL_EPOCHS


# SEED UTIL

def set_global_seed(seed: int):
    print(f"[SEED] Using seed = {seed}")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False



# OPTUNA OBJECTIVE

def objective(trial: optuna.Trial) -> float:
    """
    One Optuna trial:
      - sample hyperparameters
      - train YOLO for EPOCHS_PER_TRIAL
      - run validation on 'val' split
      - return mAP50 to maximize
    """
    # sample hyperparameters 
    seed = trial.suggest_int("seed", 1, 100000)

    # typical YOLOv8 hyperparameters to tune
    lr0        = trial.suggest_float("lr0", 1e-3, 5e-2, log=True)
    lrf        = trial.suggest_float("lrf", 0.1, 1.0)
    momentum   = trial.suggest_float("momentum", 0.8, 0.98)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    translate  = trial.suggest_float("translate", 0.0, 0.10)
    scale      = trial.suggest_float("scale", 0.0, 0.20)
    warmup_ep  = trial.suggest_int("warmup_epochs", 0, 5)

    # optimizer choice
    optimizer = trial.suggest_categorical("optimizer", ["SGD", "AdamW"])

    #  set seeds 
    set_global_seed(seed)

    #  model & run name 
    run_name = f"optuna_t{trial.number}"
    model = YOLO(WEIGHTS)

    #  train quick trial 
    print(f"\n[TRIAL {trial.number}] Training with params:")
    print(trial.params)

    train_res = model.train(
        data=DATA_YAML,
        epochs=EPOCHS_PER_TRIAL,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=4,
        pretrained=True,
        optimizer=optimizer,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        cos_lr=True,
        warmup_epochs=warmup_ep,

        # VIN-friendly augs (as before)
        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        flipud=0.0,
        fliplr=0.0,
        mosaic=0.0,
        mixup=0.0,
        degrees=0.0,
        shear=0.0,
        translate=translate,
        scale=scale,

        rect=True,
        project=SAVE_DIR,
        name=run_name,
        patience=10,
        verbose=False,
        seed=seed,
    )

    #  evaluate best weights from this trial 
    best_path = model.trainer.best
    best_model = YOLO(best_path)
    val_metrics = best_model.val(split="val", plots=False, verbose=False)

    map50 = float(val_metrics.box.map50)  # we maximize mAP50
    print(f"[TRIAL {trial.number}] mAP50 = {map50:.4f}")

    return map50


# 
# RUN OPTUNA SEARCH
# 
def run_optuna_search():
    study = optuna.create_study(
        study_name="yolo_vin_optuna",
        direction="maximize",
    )
    study.optimize(objective, n_trials=N_TRIALS)

    print("\n=== OPTUNA DONE ===")
    print("Best value (mAP50):", study.best_value)
    print("Best params:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    # Save study
    study_path = os.path.join(SAVE_DIR, "optuna_study.json")
    with open(study_path, "w") as f:
        json.dump({
            "best_value": study.best_value,
            "best_params": study.best_params,
        }, f, indent=2)
    print("Saved Optuna summary to:", study_path)

    return study.best_params

 
# FINAL TRAINING WITH BEST PARAMS
def train_final_model(best_params: dict):
    # extract
    seed        = best_params["seed"]
    lr0         = best_params["lr0"]
    lrf         = best_params["lrf"]
    momentum    = best_params["momentum"]
    weight_decay= best_params["weight_decay"]
    translate   = best_params["translate"]
    scale       = best_params["scale"]
    warmup_ep   = best_params["warmup_epochs"]
    optimizer   = best_params["optimizer"]

    set_global_seed(seed)

    final_run_name = "final_best_100ep"
    model = YOLO(WEIGHTS)

    print("\n[FINAL TRAIN] using best Optuna hyperparameters:")
    print(best_params)

    train_res = model.train(
        data=DATA_YAML,
        epochs=FINAL_EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        workers=4,
        pretrained=True,
        optimizer=optimizer,
        lr0=lr0,
        lrf=lrf,
        momentum=momentum,
        weight_decay=weight_decay,
        cos_lr=True,
        warmup_epochs=warmup_ep,

        hsv_h=0.0,
        hsv_s=0.0,
        hsv_v=0.0,
        flipud=0.0,
        fliplr=0.0,
        mosaic=0.0,
        mixup=0.0,
        degrees=0.0,
        shear=0.0,
        translate=translate,
        scale=scale,

        rect=True,
        project=SAVE_DIR,
        name=final_run_name,
        patience=25,
        save_period=10,
        seed=seed,
    )

    # get best weights path & run dir
    best_path = model.trainer.best
    run_dir   = model.trainer.save_dir

    # eval final model on val and (optionally) test
    best_model = YOLO(best_path)
    val_metrics = best_model.val(split="val")
    print("\n[FINAL VAL] metrics:")
    print(f"  mAP50-95: {val_metrics.box.map:.4f}")
    print(f"  mAP50   : {val_metrics.box.map50:.4f}")
    print(f"  mAP75   : {val_metrics.box.map75:.4f}")
    print(f"  Precision: {val_metrics.box.mp:.4f}  Recall: {val_metrics.box.mr:.4f}")

    try:
        test_metrics = best_model.val(split="test")
    except Exception as e:
        print("\n[FINAL TEST] skipped:", e)
        test_metrics = None

    # summarize metrics
    summary = {
        "best_params": best_params,
        "final_val": {
            "map50-95": float(val_metrics.box.map),
            "map50":    float(val_metrics.box.map50),
            "map75":    float(val_metrics.box.map75),
            "precision":float(val_metrics.box.mp),
            "recall":   float(val_metrics.box.mr),
        }
    }
    if test_metrics:
        summary["final_test"] = {
            "map50-95": float(test_metrics.box.map),
            "map50":    float(test_metrics.box.map50),
            "map75":    float(test_metrics.box.map75),
            "precision":float(test_metrics.box.mp),
            "recall":   float(test_metrics.box.mr),
        }

    with open(os.path.join(SAVE_DIR, "final_metrics_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)

    # copy best weights + artifacts to top level
    final_ckpt = os.path.join(SAVE_DIR, "vin_yolov8s_best_autotuned.pt")
    shutil.copy(best_path, final_ckpt)
    print("\nSaved final best weights as:", final_ckpt)

    for a in [
        "results.png", "F1_curve.png", "PR_curve.png", "P_curve.png", "R_curve.png",
        "confusion_matrix.png", "confusion_matrix_normalized.png", "results.csv"
    ]:
        src = os.path.join(run_dir, a)
        if os.path.exists(src):
            shutil.copy(src, os.path.join(SAVE_DIR, a))

    print("\nAll artifacts saved in:", SAVE_DIR)



# MAIN

if __name__ == "__main__":
    # 1) Hyperparameter + seed search
    best_params = run_optuna_search()

    # 2) Final 100-epoch training with best params
    train_final_model(best_params)

c:\JLR_VIN_2549\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2025-11-22 17:17:38,199] A new study created in memory with name: yolo_vin_optuna


[SEED] Using seed = 91905

[TRIAL 0] Training with params:
{'seed': 91905, 'lr0': 0.001803604600554006, 'lrf': 0.6772624441693993, 'momentum': 0.899937916811746, 'weight_decay': 1.1997284457834516e-05, 'translate': 0.09145987383788692, 'scale': 0.08491969410799945, 'warmup_epochs': 0, 'optimizer': 'AdamW'}
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\JLR_VIN_2549\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, ke

[I 2025-11-22 17:47:20,325] Trial 0 finished with value: 0.9845580624531709 and parameters: {'seed': 91905, 'lr0': 0.001803604600554006, 'lrf': 0.6772624441693993, 'momentum': 0.899937916811746, 'weight_decay': 1.1997284457834516e-05, 'translate': 0.09145987383788692, 'scale': 0.08491969410799945, 'warmup_epochs': 0, 'optimizer': 'AdamW'}. Best is trial 0 with value: 0.9845580624531709.


[SEED] Using seed = 35588

[TRIAL 1] Training with params:
{'seed': 35588, 'lr0': 0.0029269779658997685, 'lrf': 0.5534500255603553, 'momentum': 0.856935369873619, 'weight_decay': 0.0001725440253945299, 'translate': 0.07299359540449925, 'scale': 0.08587097857162707, 'warmup_epochs': 0, 'optimizer': 'AdamW'}
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\JLR_VIN_2549\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, ke

[I 2025-11-22 18:16:47,409] Trial 1 finished with value: 0.9440654506462446 and parameters: {'seed': 35588, 'lr0': 0.0029269779658997685, 'lrf': 0.5534500255603553, 'momentum': 0.856935369873619, 'weight_decay': 0.0001725440253945299, 'translate': 0.07299359540449925, 'scale': 0.08587097857162707, 'warmup_epochs': 0, 'optimizer': 'AdamW'}. Best is trial 0 with value: 0.9845580624531709.


...

[SEED] Using seed = 21468

[TRIAL 13] Training with params:
{'seed': 21468, 'lr0': 0.010875907549992106, 'lrf': 0.8466530227757278, 'momentum': 0.8753348932704392, 'weight_decay': 7.236986895226463e-05, 'translate': 0.035954574120761806, 'scale': 0.11915201136828309, 'warmup_epochs': 3, 'optimizer': 'SGD'}
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\JLR_VIN_2549\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, ke

[I 2025-11-23 00:10:13,512] Trial 13 finished with value: 0.9948627129938169 and parameters: {'seed': 21468, 'lr0': 0.010875907549992106, 'lrf': 0.8466530227757278, 'momentum': 0.8753348932704392, 'weight_decay': 7.236986895226463e-05, 'translate': 0.035954574120761806, 'scale': 0.11915201136828309, 'warmup_epochs': 3, 'optimizer': 'SGD'}. Best is trial 9 with value: 0.995.


[SEED] Using seed = 846

[TRIAL 14] Training with params:
{'seed': 846, 'lr0': 0.006514321763567527, 'lrf': 0.8274777314946405, 'momentum': 0.9326193123042117, 'weight_decay': 0.00011641391567382992, 'translate': 0.0602294385433915, 'scale': 0.0570391510642679, 'warmup_epochs': 4, 'optimizer': 'SGD'}
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\JLR_VIN_2549\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=1024, int8=False, iou=0.7, keras=Fa

[I 2025-11-23 00:39:37,743] Trial 14 finished with value: 0.9535159404157535 and parameters: {'seed': 846, 'lr0': 0.006514321763567527, 'lrf': 0.8274777314946405, 'momentum': 0.9326193123042117, 'weight_decay': 0.00011641391567382992, 'translate': 0.0602294385433915, 'scale': 0.0570391510642679, 'warmup_epochs': 4, 'optimizer': 'SGD'}. Best is trial 9 with value: 0.995.



=== OPTUNA DONE ===
Best value (mAP50): 0.995
Best params:
  seed: 15172
  lr0: 0.007940597486362172
  lrf: 0.8902538925876162
  momentum: 0.8712294492880323
  weight_decay: 0.0001072191024005336
  translate: 0.07828047179886817
  scale: 0.054878253792813504
  warmup_epochs: 3
  optimizer: SGD
Saved Optuna summary to: C:\JLR_VIN_2549\YOLO_trained_optuna\optuna_study.json
[SEED] Using seed = 15172

[FINAL TRAIN] using best Optuna hyperparameters:
{'seed': 15172, 'lr0': 0.007940597486362172, 'lrf': 0.8902538925876162, 'momentum': 0.8712294492880323, 'weight_decay': 0.0001072191024005336, 'translate': 0.07828047179886817, 'scale': 0.054878253792813504, 'warmup_epochs': 3, 'optimizer': 'SGD'}
Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None

In [ ]:
from ultralytics import YOLO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import os

# Import centralized config
from config import DATA_YAML, YOLO_BEST_OPTUNA, TEST_IMAGES

# Load the trained VIN model
model = YOLO(str(YOLO_BEST_OPTUNA))

# Evaluate on TEST dataset
metrics = model.val(
    data=str(DATA_YAML),  # dataset config YAML
    split="test"
)

# Show per-class metrics in dataframe
class_metrics = {
    "precision": metrics.box.p.tolist(),
    "recall": metrics.box.r.tolist(),
    "f1": metrics.box.f1.tolist(),
    "ap50": metrics.box.ap50.tolist(),
    "ap": metrics.box.ap.tolist()
}

names = list(model.names.values())[:len(class_metrics["precision"])]  # align with actual detected classes
df = pd.DataFrame(class_metrics, index=names)

print("\n Per-class results:")
print(df)

# Create and plot Confusion Matrix
# Get predictions for test set
test_results = model.predict(str(TEST_IMAGES), save=False, verbose=False)

# Extract true and predicted labels
true_labels = []
pred_labels = []

for result in test_results:
    if result.boxes is not None:
        # Get ground truth (if available in result)
        if hasattr(result, 'boxes') and result.boxes.cls is not None:
            true_labels.extend(result.boxes.cls.cpu().numpy())

        # Get predictions
        if hasattr(result, 'boxes') and result.boxes.cls is not None:
            pred_labels.extend(result.boxes.cls.cpu().numpy())

# Create confusion matrix
if true_labels and pred_labels:
    cm = confusion_matrix(true_labels, pred_labels, labels=range(len(names)))

    # Plot confusion matrix
    plt.figure(figsize=(16, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=names, yticklabels=names,
                cbar_kws={'label': 'Count'})
    plt.title('Confusion Matrix - YOLOv8 VIN Character Detection', fontsize=16, pad=20)
    plt.xlabel('Predicted Label', fontsize=14)
    plt.ylabel('True Label', fontsize=14)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    # Calculate and display accuracy metrics
    accuracy = np.trace(cm) / np.sum(cm) if np.sum(cm) > 0 else 0
    print(f"\n Overall Accuracy: {accuracy:.3f}")

    # Per-class accuracy
    class_accuracy = np.diag(cm) / np.sum(cm, axis=1)
    accuracy_df = pd.DataFrame({
        'Class': names,
        'Accuracy': class_accuracy,
        'Support': np.sum(cm, axis=1)
    })
    print("\n Per-class Accuracy:")
    print(accuracy_df)
else:
    print("Could not extract labels for confusion matrix")

# Additional performance metrics

print("\n Overall Metrics:")
print(f"mAP@0.5       : {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95  : {metrics.box.map:.3f}")
print(f"Mean Precision: {metrics.box.mp:.3f}")
print(f"Mean Recall   : {metrics.box.mr:.3f}")

# Compute overall F1 score manually as mean of per-class F1
overall_f1 = float(np.mean(metrics.box.f1))
print(f"Overall F1    : {overall_f1:.3f}")

Ultralytics 8.3.230  Python-3.11.5 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 3050 OEM, 8192MiB)
Model summary (fused): 72 layers, 11,138,355 parameters, 0 gradients, 28.5 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 3389.8675.3 MB/s, size: 728.0 KB)
val: Scanning C:\JLR_VIN_2549\test\labels.cache... 160 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 160/160 159.9Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 2.3it/s 4.3s.4ss
                   all        160       3040      0.994      0.994      0.995       0.97
                     0        160        320          1          1      0.995      0.992
                     1        111        148      0.999          1      0.995      0.963
                     2        108        144          1          1      0.995      0.963
                     3         84        102      0.999          1      0.995      0.958
                     4        

<Figure size 1600x1400 with 2 Axes>


 Overall Accuracy: 1.000

 Per-class Accuracy:
   Class  Accuracy  Support
0      0       1.0      320
1      1       1.0      148
2      2       1.0      144
3      3       1.0      103
4      4       1.0      216
5      5       1.0      109
6      6       1.0      134
7      7       1.0      204
8      8       1.0       61
9      9       1.0       61
10     A       1.0      162
11     B       1.0      446
12     C       1.0       77
13     D       NaN        0
14     E       NaN        0
15     F       1.0       32
16     G       1.0       68
17     H       NaN        0
18     J       NaN        0
19     K       NaN        0
20     L       1.0       85
21     M       1.0      176
22     P       1.0        1
23     R       NaN        0

 Overall Metrics:
mAP@0.5       : 0.995
mAP@0.5:0.95  : 0.970
Mean Precision: 0.994
Mean Recall   : 0.994
Overall F1    : 0.994


C:\Users\MURTI\AppData\Local\Temp\ipykernel_23952\2561870152.py:73: RuntimeWarning: invalid value encountered in divide
  class_accuracy = np.diag(cm) / np.sum(cm, axis=1)


In [ ]:
import os
import sys
import csv
import re
from pathlib import Path

import cv2
import numpy as np
from PySide6 import QtCore, QtGui, QtWidgets
from ultralytics import YOLO

# ============================================================================
# IMPORT CENTRALIZED CONFIG
# ============================================================================
from config import PROJECT_ROOT, YOLO_BEST_OPTUNA, CLASS_NAMES

#  CONFIG - Now uses centralized paths
DATASET_ROOT = str(PROJECT_ROOT)
MODEL_PATH   = str(YOLO_BEST_OPTUNA)

# class list (order MUST match training) - from config
NAMES = CLASS_NAMES

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".tif",".tiff",".JPG",".PNG",".BMP",".TIF",".TIFF"}

# GT from filename
_VIN_PATTERNS = [
    r'\*[A-HJ-NPR-Z0-9]{17}\*',   # *VIN*
    r'[A-HJ-NPR-Z0-9]{17}',       # VIN
]
def extract_vin_from_filename(path: str) -> str | None:
    s = os.path.splitext(os.path.basename(path.upper()))[0]
    for p in _VIN_PATTERNS:
        m = re.search(p, s)
        if m:
            v = m.group(0)
            return v if v.startswith('*') else f"*{v}*"
    return None

# core helpers
def build_vin_from_result(ultra_result, keep_star=False, conf_thres=0.35, iou=0.5):
    boxes = ultra_result.boxes
    if boxes is None or len(boxes) == 0:
        return "", []
    items = []
    for b in boxes:
        cls = int(b.cls.item())
        ch  = NAMES[cls] if 0 <= cls < len(NAMES) else '?'
        if (not keep_star) and ch == '*':
            continue
        conf = float(b.conf.item())
        if conf < conf_thres:
            continue
        x1,y1,x2,y2 = [float(v) for v in b.xyxy[0].tolist()]
        xc = (x1+x2)/2.0
        items.append((xc, ch, conf, (x1,y1,x2,y2)))
    items.sort(key=lambda t: t[0])  # left-to-right
    vin = "".join(ch for _, ch, _, _ in items)
    boxes_out = [(x1,y1,x2,y2,ch,conf) for _, ch, conf, (x1,y1,x2,y2) in items]
    return vin, boxes_out

def draw_overlay(img_bgr, boxes, pred_vin, exp_vin=None, conf_used=0.35, low_conf_delta=0.10):
    canvas = img_bgr.copy()
    low_thr = max(0.0, conf_used - low_conf_delta)
    any_wrong = False

    for i, (x1,y1,x2,y2,ch,conf) in enumerate(boxes):
        correct = None
        if exp_vin is not None:
            if i < len(exp_vin):
                correct = (ch == exp_vin[i])
            else:
                correct = False
        # default (correct)
        color_box, color_text = (0,255,0), (0,0,255)
        if correct is False:
            color_box, color_text = (0,0,255), (255,255,255)
            any_wrong = True
        if conf < low_thr:  # low-confidence → orange
            color_box, color_text = (0,140,255), (255,255,255)

        p1 = (int(x1), int(y1)); p2 = (int(x2), int(y2))
        cv2.rectangle(canvas, p1, p2, color_box, 2)
        cv2.putText(canvas, ch, (p1[0], max(10, p1[1]-6)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, color_text, 2, cv2.LINE_AA)

    # headers
    y0 = 30
    title_ok = (exp_vin is not None and pred_vin == exp_vin and not any_wrong)
    title_color = (0,255,0) if title_ok else (0,0,255)
    cv2.putText(canvas, f"Pred: {pred_vin}", (10, y0),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, title_color, 2, cv2.LINE_AA)
    if exp_vin:
        cv2.putText(canvas, f"Exp : {exp_vin}", (10, y0+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,255), 2, cv2.LINE_AA)
    return canvas

def bgr2pixmap(img_bgr):
    h,w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    qimg = QtGui.QImage(img_rgb.data, w, h, 3*w, QtGui.QImage.Format_RGB888)
    return QtGui.QPixmap.fromImage(qimg)

# threaded predictor
class InferenceWorker(QtCore.QThread):
    progress = QtCore.Signal(int)                 # percent
    item_done = QtCore.Signal(dict)               # per-image result
    finished  = QtCore.Signal()

    def __init__(self, model: YOLO, paths, out_dir, conf, keep_star, parent=None):
        super().__init__(parent)
        self.model = model
        self.paths = paths
        self.out_dir = Path(out_dir) if out_dir else None
        self.conf = float(conf)
        self.keep_star = bool(keep_star)
        self._stop = False

    def stop(self):
        self._stop = True

    def run(self):
        csv_f, csv_w = None, None
        if self.out_dir:
            (self.out_dir / "annotated").mkdir(parents=True, exist_ok=True)
            csv_path = self.out_dir / "predictions.csv"
            csv_f = open(csv_path, "w", newline="", encoding="utf-8")
            csv_w = csv.writer(csv_f)
            csv_w.writerow(["image","expected","prediction","status"])

        n = max(1, len(self.paths))
        for i, path in enumerate(self.paths, start=1):
            if self._stop:
                break
            try:
                img = cv2.imread(str(path))
                if img is None:
                    raise RuntimeError(f"Failed to read: {path}")

                img_in = cv2.convertScaleAbs(img, alpha=1.2, beta=0)
                result = self.model.predict(img_in, conf=self.conf, iou=0.5, imgsz=1024, verbose=False)[0]
                pred_str, boxes = build_vin_from_result(result, keep_star=self.keep_star, conf_thres=self.conf)

                exp_vin = extract_vin_from_filename(path)
                if len(boxes) == 0 or not pred_str:
                    status = "no-detect"
                elif exp_vin and pred_str == exp_vin:
                    status = "correct"
                else:
                    status = "wrong"

                overlay = draw_overlay(img, boxes, pred_str, exp_vin=exp_vin, conf_used=self.conf)

                if self.out_dir and csv_w:
                    out_img = self.out_dir / "annotated" / (Path(path).stem + "_pred.jpg")
                    cv2.imwrite(str(out_img), overlay)
                    csv_w.writerow([str(path), exp_vin or "", pred_str, status])

                self.item_done.emit({
                    "path": str(path),
                    "pred": pred_str,
                    "exp":  exp_vin or "",
                    "status": status,
                    "overlay": overlay
                })
            except Exception as e:
                self.item_done.emit({
                    "path": str(path),
                    "pred": f"[ERROR] {e}",
                    "exp":  "",
                    "status": "error",
                    "overlay": None
                })

            self.progress.emit(int(i*100/n))

        if csv_f:
            csv_f.close()
        self.finished.emit()

#  metrics dialog (pie + F1)
class MetricsDialog(QtWidgets.QDialog):
    def __init__(self, tp, fp, fn, parent=None):
        super().__init__(parent)
        self.setWindowTitle("Batch Metrics")
        self.resize(720, 520)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall    = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2*precision*recall)/(precision+recall) if (precision+recall) else 0.0

        from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
        from matplotlib.figure import Figure

        fig = Figure(figsize=(6,5), tight_layout=True)
        canvas = FigureCanvas(fig)
        ax = fig.add_subplot(111)

        labels = [f"Correct (TP): {tp}", f"Wrong (FP): {fp}", f"No-detect (FN): {fn}"]
        sizes  = [tp, fp, fn]
        colors = ["#2ca02c", "#d62728", "#ff7f0e"]

        wedges, *_ = ax.pie(
            sizes, labels=None, autopct=lambda p: f"{p:.1f}%" if p >= 0.1 else "",
            startangle=90, colors=colors, textprops={'color': "black"}
        )
        ax.legend(wedges, labels, loc="upper center", bbox_to_anchor=(0.5, 1.12), ncol=3, frameon=False)
        ax.axis('equal')
        ax.set_title(f"No-detect (FN): {fn}   Wrong (FP): {fp}")

        footer = QtWidgets.QLabel(
            f"<b>Images scored (with GT)</b>: {tp+fp+fn}<br>"
            f"<b>Precision</b>: {precision:.3f} &nbsp;&nbsp; "
            f"<b>Recall</b>: {recall:.3f} &nbsp;&nbsp; "
            f"<b>F1</b>: {f1:.3f}"
        )
        footer.setAlignment(QtCore.Qt.AlignLeft)

        btn = QtWidgets.QPushButton("Close")
        btn.clicked.connect(self.accept)

        lay = QtWidgets.QVBoxLayout(self)
        lay.addWidget(canvas)
        lay.addWidget(footer)
        lay.addWidget(btn, alignment=QtCore.Qt.AlignRight)

# GUI
class MainWindow(QtWidgets.QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("VIN OCR – OVST (Optuna Model)")
        self.resize(1200, 780)

        # load model once
        self.model = YOLO(MODEL_PATH)

        # left: preview + controls
        self.imageLabel = QtWidgets.QLabel("Preview")
        self.imageLabel.setAlignment(QtCore.Qt.AlignCenter)
        self.imageLabel.setMinimumSize(680,520)
        self.imageLabel.setStyleSheet("QLabel { background: #111; color: #999; }")

        self.btnSingle  = QtWidgets.QPushButton("Open Image(s)…")
        self.btnFolder  = QtWidgets.QPushButton("Open Folder…")
        self.btnSaveDir = QtWidgets.QPushButton("Set Output Folder…")
        self.btnClear   = QtWidgets.QPushButton("Clear Results")
        self.btnMetrics = QtWidgets.QPushButton("Show Metrics (Pie/F1)")
        self.btnExitApp = QtWidgets.QPushButton("Program Exit (reset kernel)")

        self.chkKeepStar = QtWidgets.QCheckBox("Keep STAR (*) in VIN")
        self.chkKeepStar.setChecked(True)

        self.confSpin = QtWidgets.QDoubleSpinBox()
        self.confSpin.setRange(0.0, 1.0)
        self.confSpin.setSingleStep(0.05)
        self.confSpin.setValue(0.35)
        self.confSpin.setDecimals(2)

        self.progress = QtWidgets.QProgressBar()
        self.progress.setRange(0,100)
        self.progress.setValue(0)

        left = QtWidgets.QVBoxLayout()
        left.addWidget(self.imageLabel, 4)
        left.addWidget(self.progress)

        ctrlGrid = QtWidgets.QGridLayout()
        ctrlGrid.addWidget(QtWidgets.QLabel("Confidence:"), 0,0)
        ctrlGrid.addWidget(self.confSpin, 0,1)
        ctrlGrid.addWidget(self.chkKeepStar, 0,2)
        ctrlGrid.addWidget(self.btnSingle, 1,0)
        ctrlGrid.addWidget(self.btnFolder, 1,1)
        ctrlGrid.addWidget(self.btnSaveDir,1,2)
        ctrlGrid.addWidget(self.btnClear, 2,0)
        ctrlGrid.addWidget(self.btnMetrics, 2,1)
        ctrlGrid.addWidget(self.btnExitApp, 2,2)  # << EXIT BUTTON
        left.addLayout(ctrlGrid)

        # right: results table (3 cols)
        self.table = QtWidgets.QTableWidget(0, 3)
        self.table.setHorizontalHeaderLabels(["Image","Prediction","Status"])
        self.table.horizontalHeader().setStretchLastSection(True)
        self.table.setSelectionBehavior(QtWidgets.QAbstractItemView.SelectRows)
        self.table.setEditTriggers(QtWidgets.QAbstractItemView.NoEditTriggers)

        # root layout
        right = QtWidgets.QVBoxLayout()
        right.addWidget(self.table, 1)

        root = QtWidgets.QWidget()
        layout = QtWidgets.QHBoxLayout(root)
        layout.addLayout(left, 2)
        layout.addLayout(right, 1)
        self.setCentralWidget(root)

        # state
        self.output_dir = None
        self.last_overlays = {}  # path -> overlay image
        self.rows = []           # accumulate results: dict(path,pred,exp,status)
        self.worker: InferenceWorker | None = None

        # signals
        self.btnSingle.clicked.connect(self.open_images)
        self.btnFolder.clicked.connect(self.open_folder)
        self.btnClear.clicked.connect(self.clear_results)
        self.btnSaveDir.clicked.connect(self.pick_output_dir)
        self.btnMetrics.clicked.connect(self.show_metrics)
        self.btnExitApp.clicked.connect(self.exit_and_reset_kernel)  # << hook
        self.table.itemSelectionChanged.connect(self.on_row_selected)

    # clean worker stop for all exit paths
    def _stop_worker(self):
        if self.worker and self.worker.isRunning():
            self.worker.stop()
            self.worker.wait(2000)

    # make the window X button safe too
    def closeEvent(self, e: QtGui.QCloseEvent):
        self._stop_worker()
        e.accept()

    # Exit button: stop worker, close GUI, and restart kernel if in notebook/VS Code
    def exit_and_reset_kernel(self):
        self._stop_worker()
        self.close()

        # Try to restart kernel if running inside IPython (Jupyter/VS Code)
        try:
            from IPython import get_ipython
            ip = get_ipython()
            if ip is not None and hasattr(ip, "kernel"):
                # Restart the kernel so you can re-run the cell cleanly
                ip.kernel.do_shutdown(restart=True)
                return
        except Exception:
            pass

        # Fallback: just quit the app/process
        app = QtWidgets.QApplication.instance()
        if app is not None:
            app.quit()
        # ensure process exit if still alive
        QtCore.QTimer.singleShot(100, lambda: os._exit(0))

    # helper
    def pick_output_dir(self):
        d = QtWidgets.QFileDialog.getExistingDirectory(self, "Select Output Folder", DATASET_ROOT)
        if d:
            self.output_dir = d

    def clear_results(self):
        self.table.setRowCount(0)
        self.imageLabel.setPixmap(QtGui.QPixmap())
        self.progress.setValue(0)
        self.last_overlays.clear()
        self.rows.clear()

    def on_row_selected(self):
        rows = self.table.selectionModel().selectedRows()
        if not rows:
            return
        row = rows[0].row()
        path = self.table.item(row, 0).text()
        overlay = self.last_overlays.get(path)
        if overlay is not None:
            self.imageLabel.setPixmap(bgr2pixmap(overlay).scaled(
                self.imageLabel.size(), QtCore.Qt.KeepAspectRatio, QtCore.Qt.SmoothTransformation))

    # actions
    def open_images(self):
        files, _ = QtWidgets.QFileDialog.getOpenFileNames(
            self, "Select Image(s)", str(Path.home()),
            "Images (*.png *.jpg *.jpeg *.bmp *.tif *.tiff);;All Files (*)")
        paths = [p for p in files if Path(p).suffix.lower() in {e.lower() for e in IMG_EXTS}]
        if paths:
            self.run_inference(paths)

    def open_folder(self):
        d = QtWidgets.QFileDialog.getExistingDirectory(self, "Select Folder", str(Path.home()))
        if not d:
            return
        exts = {e.lower() for e in IMG_EXTS}
        paths = [str(p) for p in Path(d).rglob("*") if p.is_file() and p.suffix.lower() in exts]
        if paths:
            self.run_inference(paths)

    def run_inference(self, paths):
        self.progress.setValue(0)
        self.table.setRowCount(0)
        self.last_overlays.clear()
        self.rows.clear()

        conf = float(self.confSpin.value())
        keep_star = self.chkKeepStar.isChecked()

        # stop previous worker if any
        self._stop_worker()

        self.worker = InferenceWorker(self.model, paths, self.output_dir, conf, keep_star)
        self.worker.progress.connect(self.progress.setValue)
        self.worker.item_done.connect(self.add_result_row)
        self.worker.finished.connect(lambda: QtWidgets.QMessageBox.information(self, "Done", "Inference completed."))
        self.worker.start()

    @QtCore.Slot(dict)
    def add_result_row(self, res):
        self.rows.append(res)

        row = self.table.rowCount()
        self.table.insertRow(row)
        self.table.setItem(row, 0, QtWidgets.QTableWidgetItem(res["path"]))
        self.table.setItem(row, 1, QtWidgets.QTableWidgetItem(res["pred"]))

        status_text = res.get("status","")
        status_item = QtWidgets.QTableWidgetItem(status_text)
        if status_text == "correct":
            status_item.setForeground(QtGui.QBrush(QtGui.QColor(0,150,0)))
        elif status_text == "wrong":
            status_item.setForeground(QtGui.QBrush(QtGui.QColor(200,0,0)))
        elif status_text == "no-detect":
            status_item.setForeground(QtGui.QBrush(QtGui.QColor(255,140,0)))
        else:
            status_item.setForeground(QtGui.QBrush(QtGui.QColor(150,150,150)))
        self.table.setItem(row, 2, status_item)

        if res["overlay"] is not None:
            self.last_overlays[res["path"]] = res["overlay"]
            if row == 0:
                self.imageLabel.setPixmap(bgr2pixmap(res["overlay"]).scaled(
                    self.imageLabel.size(), QtCore.Qt.KeepAspectRatio, QtCore.Qt.SmoothTransformation))

    def show_metrics(self):
        tp = sum(1 for r in self.rows if r.get("status") == "correct")
        fp = sum(1 for r in self.rows if r.get("status") == "wrong")
        fn = sum(1 for r in self.rows if r.get("status") == "no-detect")
        dlg = MetricsDialog(tp, fp, fn, self)
        dlg.exec()

#  main
def main():
    app = QtWidgets.QApplication(sys.argv)
    w = MainWindow()
    w.show()
    sys.exit(app.exec())

if __name__ == "__main__":
    main()

c:\JLR_VIN_2549\.venv\Lib\site-packages\matplotlib\axes\_axes.py:3368: RuntimeWarning: invalid value encountered in divide
  x = x / sx


ValueError: cannot convert float NaN to integer

SystemExit: 0

c:\JLR_VIN_2549\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


: 